In [7]:
import os
import bs4
from dotenv import load_dotenv

from langchain_classic.retrievers import MultiQueryRetriever
from langchain_chroma import Chroma
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.tracers import LangChainTracer
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings

# Logical Routing

In [2]:
from typing import Literal
from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate
from langchain_groq import ChatGroq

# Data model
class RouteQuery(BaseModel):
    """Route a user query to the most relevant datasource."""

    datasource: Literal["python_docs", "js_docs", "golang_docs"] = Field(
        ...,
        description="Given a user question choose which datasource would be most relevant for answering their question",
    )

# LLM with function call 
llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0)
structured_llm = llm.with_structured_output(RouteQuery)

# Prompt 
system = """You are an expert at routing a user question to the appropriate data source.

Based on the programming language the question is referring to, route it to the relevant data source."""

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system),
        ("human", "{question}"),
    ]
)

# Define router 
router = prompt | structured_llm

In [4]:
question = """Why doesn't the following code work:

from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages(["human", "speak in {language}"])
prompt.invoke("french")
"""

result = router.invoke({"question": question})
print(result)

datasource='python_docs'


# Semantic Routing

In [15]:
from langchain_community.utils.math import cosine_similarity
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_groq import ChatGroq

# Two prompts
physics_template = """You are a very smart physics professor. \
You are great at answering questions about physics in a concise and easy to understand manner. \
When you don't know the answer to a question you admit that you don't know.

Here is a question:
{query}"""

math_template = """You are a very good mathematician. You are great at answering math questions. \
You are so good because you are able to break down hard problems into their component parts, \
answer the component parts, and then put them together to answer the broader question.

Here is a question:
{query}"""

llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0)

# Embed prompts
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)


prompt_templates = [physics_template, math_template]
prompt_embeddings = embeddings.embed_documents(prompt_templates)

def prompt_router(inputs: dict) -> str:
    """Route a query to the most relevant prompt template."""

    query = inputs["query"]
    query_embedding = embeddings.embed_query(query)

    similarities = cosine_similarity([query_embedding], prompt_embeddings)[0]
    best_prompt_index = similarities.argmax()
    
    return prompt_templates[best_prompt_index].format(query=query)

chain = (
    {"query": RunnablePassthrough()}
    | RunnableLambda(prompt_router)
    | llm
    | StrOutputParser()
)

tracer = LangChainTracer(project_name="multi-query-rag")

print(
    chain.invoke(
        "What is the speed of light?",
        config={"callbacks": [tracer]},
    )
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

The speed of light in a vacuum is a fundamental constant:

\[
c \;=\; 299\,792\,458 \text{ meters per second (m/s)}.
\]

It is the same for all observers, regardless of their motion, and it sets the ultimate speed limit for information and matter in the universe.
